In [ ]:
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

logging.getLogger("transformers").setLevel(logging.ERROR)

import sys
sys.path.append("../../utils/")

from utils import *

In [ ]:
# =========================
# CONFIGURACIÓN
# =========================

k = 1
n_folds = 5

dataset_name = "pd_CIC18__RUS_SMOTE__v1"

batch_size_pred = 16
source_max_token_len = 150
target_max_token_len = 3
use_gpu = False   # CPU

In [ ]:
# =========================
# RUTAS
# =========================

BASE_DIR = Path("../../../").resolve()

ruta_base_dataset = BASE_DIR / "02_datasets" / "processed" / dataset_name

output_path = (
    BASE_DIR
    / "04_experimentos"
    / "modelos"
    / f"{dataset_name}__outputs"
    / f"{dataset_name}__outputs__{k}_{n_folds}"
)

ruta_test = output_path / "test_final"
ruta_test.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# CARGA TEST
# =========================

nombre_test = f"{dataset_name}__test.csv"

df_test = cargar_dataset(nombre_test, ruta_base_dataset)
# o si prefieres:
# df_test = pd.read_csv(ruta_base_dataset / nombre_test)

df_test["source_text"] = df_test["source_text"].astype(str)
df_test["target_text"] = df_test["target_text"].astype(str)

print("Test shape:", df_test.shape)
df_test.head()

In [ ]:
# =========================
# CARGA MODELO FINAL
# =========================

model = SimpleT5Wrapper()
model.from_pretrained("t5", output_path, use_gpu=use_gpu)

print("Modelo cargado desde:", output_path)

In [ ]:
# =========================
# PREDICCIÓN
# =========================

test_texts = df_test["source_text"].tolist()
y_true = df_test["target_text"].tolist()

inicio = time.time()

y_pred = model.predict(
    test_texts,
    batch_size=batch_size_pred,
    source_max_token_len=source_max_token_len,
    target_max_token_len=target_max_token_len
)

fin = time.time()

tiempo_min = (fin - inicio) / 60

print(f"Tiempo de predicción: {tiempo_min:.2f} min")
print("Número de predicciones:", len(y_pred))

import re
from collections import Counter

# =========================
# LIMPIEZA Y DIAGNÓSTICO
# =========================

valid_labels = sorted(set(y_true), key=lambda x: int(x))
y_pred_str = [str(p).strip() for p in y_pred]

def limpiar_prediccion(pred, valid_labels):
    pred = str(pred).strip()

    if pred in valid_labels:
        return pred

    encontrados = re.findall(r"\d+", pred)
    for e in encontrados:
        if e in valid_labels:
            return e

    return "INVALID"

y_pred_final = [limpiar_prediccion(p, valid_labels) for p in y_pred_str]

# diagnóstico
n_invalid = sum(p == "INVALID" for p in y_pred_final)

print("=== DIAGNÓSTICO ===")
print(f"Inválidas: {n_invalid}")
print(f"% inválidas: {n_invalid / len(y_pred_final):.4%}")

# guardar ejemplos si hay
if n_invalid > 0:
    df_debug = pd.DataFrame({
        "target_text": y_true,
        "prediction_raw": y_pred_str,
        "prediction_final": y_pred_final
    })

    df_debug = df_debug[df_debug["prediction_final"] == "INVALID"]

    print(df_debug.head(10))

    df_debug.to_csv(
        ruta_test / f"predicciones_invalidas_test_fold_{k}.csv",
        index=False
    )

In [ ]:
# =========================
# MÉTRICAS GLOBALES
# =========================

accuracy = accuracy_score(y_true, y_pred_final)

precision_macro = precision_score(y_true, y_pred_final, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred_final, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred_final, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred_final, average="weighted", zero_division=0)
recall_weighted = recall_score(y_true, y_pred_final, average="weighted", zero_division=0)
f1_weighted = f1_score(y_true, y_pred_final, average="weighted", zero_division=0)

mcc = matthews_corrcoef(y_true, y_pred_final)

print("\n=== MÉTRICAS TEST ===")
print(f"Accuracy           : {accuracy:.6f}")
print(f"Precision macro    : {precision_macro:.6f}")
print(f"Recall macro       : {recall_macro:.6f}")
print(f"F1-score macro     : {f1_macro:.6f}")
print(f"Precision weighted : {precision_weighted:.6f}")
print(f"Recall weighted    : {recall_weighted:.6f}")
print(f"F1-score weighted  : {f1_weighted:.6f}")
print(f"MCC                : {mcc:.6f}")
print(f"Tiempo (min)       : {tiempo_min:.2f}")

In [ ]:
# =========================
# GUARDAR MÉTRICAS
# =========================

df_metricas = pd.DataFrame([{
    "fold": k,
    "modelo": output_path.name,
    "accuracy": accuracy,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "precision_weighted": precision_weighted,
    "recall_weighted": recall_weighted,
    "f1_weighted": f1_weighted,
    "mcc": mcc,
    "tiempo_min": tiempo_min
}])

df_metricas.to_csv(ruta_test / f"metricas_test_fold_{k}.csv", index=False)
df_metricas

In [ ]:
# =========================
# GUARDAR PREDICCIONES
# =========================

df_preds = pd.DataFrame({
    "source_text": test_texts,
    "target_text": y_true,
    "prediction": y_pred_final
})

df_preds["correct"] = df_preds["target_text"] == df_preds["prediction"]

df_preds.to_csv(ruta_test / f"predicciones_test_fold_{k}.csv", index=False)
df_preds.head()

In [ ]:
# =========================
# CLASSIFICATION REPORT
# =========================

report_dict = classification_report(
    y_true,
    y_pred_final,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv(ruta_test / f"classification_report_test_fold_{k}.csv")

df_report

In [ ]:
# =========================
# MATRIZ DE CONFUSIÓN
# =========================

labels_sorted = sorted(set(y_true), key=lambda x: int(x))

if "INVALID" in y_pred_final:
    labels_sorted = labels_sorted + ["INVALID"]

cm = confusion_matrix(y_true, y_pred_final, labels=labels_sorted)
df_cm = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)

df_cm.to_csv(ruta_test / f"confusion_matrix_test_fold_{k}.csv")
df_cm

In [ ]:
# =========================
# MATRIZ DE CONFUSIÓN NORMALIZADA
# =========================

cm_normalized = confusion_matrix(
    y_true,
    y_pred_final,
    labels=labels_sorted,
    normalize="true"
)

df_cm_normalized = pd.DataFrame(
    cm_normalized,
    index=labels_sorted,
    columns=labels_sorted
)

df_cm_normalized.to_csv(ruta_test / f"confusion_matrix_normalized_test_fold_{k}.csv")
df_cm_normalized

In [ ]:
# =========================
# GRÁFICA: MATRIZ DE CONFUSIÓN
# =========================

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=90, colorbar=False)
plt.title(f"Matriz de confusión - fold {k}")
plt.tight_layout()
plt.savefig(ruta_test / f"matriz_confusion_fold_{k}.png", dpi=300)
plt.show()

In [ ]:
# =========================
# GRÁFICA: MATRIZ DE CONFUSIÓN NORMALIZADA
# =========================

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=labels_sorted)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=90, values_format=".2f", colorbar=False)
plt.title(f"Matriz de confusión normalizada - fold {k}")
plt.tight_layout()
plt.savefig(ruta_test / f"matriz_confusion_normalizada_fold_{k}.png", dpi=300)
plt.show()

In [ ]:
# =========================
# GRÁFICA: MÉTRICAS GLOBALES
# =========================

metric_names = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "mcc"
]

metric_values = [
    accuracy,
    precision_macro,
    recall_macro,
    f1_macro,
    precision_weighted,
    recall_weighted,
    f1_weighted,
    mcc
]

plt.figure(figsize=(12, 6))
plt.bar(metric_names, metric_values)
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Valor")
plt.title(f"Métricas globales - fold {k}")
plt.tight_layout()
plt.savefig(ruta_test / f"metricas_globales_fold_{k}.png", dpi=300)
plt.show()

In [ ]:
# =========================
# RESUMEN JSON
# =========================

metricas_dict = {
    "fold": k,
    "modelo": output_path.name,
    "accuracy": float(accuracy),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "precision_weighted": float(precision_weighted),
    "recall_weighted": float(recall_weighted),
    "f1_weighted": float(f1_weighted),
    "mcc": float(mcc),
    "tiempo_min": float(tiempo_min)
}

import json
with open(ruta_test / f"metricas_test_fold_{k}.json", "w", encoding="utf-8") as f:
    json.dump(metricas_dict, f, indent=4)